# AEM4L5 · Notebook 1 — LoRA, PEFT y adaptación eficiente

**Clase:** Arquitecturas avanzadas de adaptación.

Este notebook es **teórico-práctico**: primero entendés el problema, después ves el código mínimo.

## 0. Setup tecnico

Instalamos solo `langchain-core` (no requiere API key). `langchain-openai` es **opcional**.

Toda la clase corre **sin claves**: simulamos el modelo con `RunnableLambda`.

In [ ]:
# Dependencia base (no requiere API key):
!pip install -q langchain-core

# Opcional, solo si quisieras usar OpenAI real (no es necesario para esta clase):
# !pip install -q langchain-openai

In [ ]:
import time
import asyncio
import cProfile
import pstats
import io
import re
from typing import List, Dict, Any

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel


def fake_llm_response(prompt_value):
    # Simulador de modelo: evita depender de APIs pagas.
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    return f"Respuesta simulada del modelo para:\n{text[:300]}"


fake_llm = RunnableLambda(fake_llm_response)
print("Setup listo. Modelo simulado disponible como fake_llm.")

## 1. Objetivo del notebook

Que puedas entender qué significa **adaptar un modelo grande sin reentrenarlo completo**, y puedas explicar:

- qué es un modelo base y qué es fine-tuning;
- qué es full fine-tuning y qué es PEFT;
- qué es LoRA, qué significa congelar parámetros y qué es un adaptador;
- cuándo conviene usar LoRA;
- qué rol cumple **LangChain** en un sistema que usa modelos adaptados.

## 2. Mapa visual del tema

```mermaid
flowchart LR
    A[Modelo base preentrenado] --> B[Congelar pesos]
    B --> C[Agregar adaptador LoRA]
    C --> D[Entrenar solo adaptador]
    D --> E[Evaluar]
    E --> F[Versionar adaptador]
    F --> G[Usar en produccion]
    A -.-> A1[Conocimiento general]
    C -.-> C1[Pocos parametros entrenables]
    G -.-> G1[Modelo base + adaptador especifico]
```

## 3. Glosario

| Término | Explicación clara |
|---|---|
| Modelo base | Modelo preentrenado con conocimiento general. |
| Fine-tuning | Proceso de adaptar un modelo con datos específicos. |
| Full fine-tuning | Ajuste donde se modifican todos o casi todos los parámetros. |
| PEFT | Parameter-Efficient Fine-Tuning: adaptar entrenando pocos parámetros. |
| LoRA | Low-Rank Adaptation: técnica PEFT que agrega adaptadores pequeños entrenables. |
| Adaptador | Módulo liviano que modifica el comportamiento del modelo base. |
| Parámetros congelados | Pesos que NO se actualizan durante el entrenamiento. |
| Parámetros entrenables | Pesos que SÍ cambian durante el entrenamiento. |
| Rank | Dimensión interna del adaptador LoRA. |
| Alpha | Factor de escala que controla el impacto del adaptador. |
| Dropout | Regularización para reducir overfitting. |
| Overfitting | Cuando el modelo memoriza datos y generaliza mal. |
| Dataset de dominio | Datos específicos de un área: legal, salud, soporte, finanzas. |

## 4. Teoría explicada paso a paso

### 4.1. El problema

En producción, muchas empresas necesitan adaptar modelos a dominios específicos: contratos legales, historias clínicas, soporte técnico, tono de marca, políticas internas o documentación financiera.

El problema es que **entrenar todo el modelo puede ser caro** en memoria, tiempo, GPU, almacenamiento, operación y versionado.

### 4.2. Full fine-tuning

Se actualizan todos o casi todos los pesos del modelo.

- **Ventajas:** mayor control; sirve para cambios profundos; útil en investigación o dominios muy distintos.
- **Desventajas:** alto costo; más memoria; más riesgo de sobreajuste; artefactos grandes; difícil tener muchas variantes por cliente.

### 4.3. PEFT

**Parameter-Efficient Fine-Tuning.** La idea: *adaptar el modelo entrenando solo una pequeña fracción de parámetros.* Permite reducir costo, iterar más rápido, guardar adaptadores pequeños, compartir un mismo modelo base y personalizar por dominio/cliente.

### 4.4. LoRA

LoRA es una técnica dentro de PEFT. La idea conceptual:

```text
Modelo base congelado
+
Adaptador LoRA entrenable
=
Modelo adaptado al dominio
```

El modelo base conserva su conocimiento general; el adaptador orienta el comportamiento hacia una tarea específica.

## 5. Tabla comparativa

| Aspecto | Full fine-tuning | LoRA / PEFT |
|---|---|---|
| Parámetros entrenables | Todos o casi todos | Fracción pequeña |
| Modelo base | Se modifica | Se congela |
| Memoria de entrenamiento | Alta | Menor |
| Almacenamiento por variante | Alto | Bajo |
| Iteración | Más lenta | Más rápida |
| Variantes por cliente | Costoso | Más manejable |
| Control total | Mayor | Menor |
| Ideal para | Cambios profundos | Adaptación eficiente |

## 6. Gráfico Mermaid: LangChain + modelo adaptado

```mermaid
flowchart LR
    A[Usuario] --> B[LangChain Chain]
    B --> C{Router de dominio}
    C --> D[Adaptador legal]
    C --> E[Adaptador salud]
    C --> F[Adaptador soporte]
    D --> G[Modelo base compartido]
    E --> G
    F --> G
    G --> H[Respuesta adaptada]
```

## 7. Mini ejemplo conceptual: ¿qué fracción se entrena?

Un modelo base puede tener miles de millones de parámetros, pero el adaptador LoRA entrena solo unos millones. Veámoslo con números.

In [ ]:
modelo_base_parametros = 7_000_000_000
adaptador_lora_parametros = 8_000_000

porcentaje_entrenable = adaptador_lora_parametros / modelo_base_parametros * 100

print(f"Parametros del modelo base: {modelo_base_parametros:,}")
print(f"Parametros del adaptador LoRA: {adaptador_lora_parametros:,}")
print(f"Porcentaje entrenable: {porcentaje_entrenable:.4f}%")

**Interpretación esperada:** el adaptador representa una fracción muy pequeña del modelo base. Ese es el motivo por el cual LoRA puede reducir el costo de adaptación.

## 8. Código Python + LangChain

> **Importante:** LangChain **no entrena** el adaptador LoRA. LoRA/PEFT se entrena con librerías como `transformers`, `peft` o `accelerate`. LangChain **consume** un modelo ya adaptado y **orquesta** prompts, chains, routing y endpoints.

Acá simulamos un endpoint legal ya adaptado con `RunnableLambda`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

def fake_legal_adapter(prompt_value):
    text = prompt_value.to_string()
    return (
        "Respuesta simulada usando adaptador legal:\n\n"
        "Resumen juridico: el documento establece obligaciones, plazos y partes.\n\n"
        f"Prompt recibido:\n{text[:300]}"
    )

legal_model = RunnableLambda(fake_legal_adapter)

prompt = ChatPromptTemplate.from_template(
    "Analiza el siguiente documento con tono legal:\n\n{documento}"
)

legal_chain = prompt | legal_model

respuesta = legal_chain.invoke({
    "documento": "Contrato de prestacion de servicios entre la Parte A y la Parte B..."
})
print(respuesta)

En una arquitectura real el flujo sería:

```text
LangChain -> Prompt / Chain / Router -> Endpoint o modelo cargado -> Modelo base + adaptador LoRA
```

## 9. Ejercicio guiado 1 — Elegir técnica de adaptación

Para cada caso, decidí: **LoRA/PEFT**, **full fine-tuning** o **estrategia híbrida**.

| Caso | Descripción |
|---|---|
| A | Startup con una GPU compartida quiere adaptar un resumidor a reportes médicos. |
| B | Empresa global quiere cambiar profundamente un modelo para un idioma con baja cobertura. |
| C | SaaS B2B quiere personalizar tono y formato para 80 clientes. |
| D | Laboratorio necesita máximo control sobre todos los pesos para investigación. |

In [ ]:
# Completa el diccionario con tu decision (LoRA, Full o Hibrida) y ejecuta para comparar.
mi_respuesta = {
    "A": "LoRA / PEFT",
    "B": "Hibrida o full fine-tuning",
    "C": "LoRA / PEFT",
    "D": "Full fine-tuning",
}

solucion = {
    "A": ("LoRA / PEFT", "Recursos limitados y adaptacion de dominio."),
    "B": ("Hibrida o full fine-tuning", "Cambio profundo y gran volumen de datos."),
    "C": ("LoRA / PEFT", "Muchas variantes pequenas por cliente."),
    "D": ("Full fine-tuning", "Necesidad de control total experimental."),
}

for caso, (esperado, motivo) in solucion.items():
    ok = "OK" if mi_respuesta.get(caso) == esperado else "REVISAR"
    print(f"[{ok}] Caso {caso}: esperado={esperado} | {motivo}")

## 10. Ejercicio guiado 2 — Metadatos de un adaptador

Un adaptador no es solo el archivo de pesos: necesita **metadatos** para usarlo bien.

In [ ]:
adapter_metadata = {
    "nombre": "adapter_legal_v1",
    "modelo_base": "mistral-7b-base",
    "dominio": "legal",
    "dataset": "contratos_y_demandas_v1",
    "fecha_entrenamiento": "2026-06-30",
    "estado": "testing",
    "rank": 8,
    "alpha": 16,
    "dropout": 0.05,
    "metricas": {"accuracy_eval": 0.82, "latencia_ms": 940},
}
adapter_metadata

### Preguntas de interpretación

1. ¿Por qué es importante guardar el **modelo base compatible**?
2. ¿Qué pasaría si cargo el **adaptador incorrecto**?
3. ¿Por qué hay que **versionar** los adaptadores?
4. ¿Qué **métrica técnica** y qué **métrica de negocio** agregarías?

## 11. Errores comunes

- Creer que LoRA siempre reemplaza al full fine-tuning.
- No evaluar el adaptador con datos no vistos.
- No versionar el adaptador.
- No registrar el modelo base compatible.
- Confundir “entrenar barato” con “tener buen producto”.
- Decir que LangChain entrena LoRA directamente.

## 12. Cierre conceptual

La pregunta de arquitectura no es *“qué técnica está de moda”*, sino qué estrategia permite **adaptar con calidad, costo controlado y velocidad de iteración**. LoRA/PEFT brilla con múltiples dominios o clientes y recursos limitados; full fine-tuning, cuando el cambio es profundo y hay datos y presupuesto. LangChain es la capa que **orquesta** esos modelos ya adaptados.